In [ ]:
# -- Imports
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing
from gymnasium.wrappers import FrameStackObservation

import ale_py
from ale_py import ALEInterface

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from Models.GumbelModels.gumbel_MCTS import GumbelMuZero
from Models.abstract_model import AbstractNetwork
from Models.MCTS_Util import ValueEncoder, inverse_contractive_mapping

import config

In [ ]:
# -- Load Config
model_config = config.ModelConfig()
config.POLICY_HEAD_SIZE = 17
config.DEVICE = "cpu"
config.BATCH_SIZE = 8
config.STARTING_EPISODE = 0

In [ ]:
# -- Setup Environment
gym.register_envs(ale_py)


def breakout_env():
    env = gym.make("ALE/Breakout-v5", frameskip=1)
    env = AtariPreprocessing(
        env,
        noop_max=10,
        frame_skip=4,
        terminal_on_life_loss=True,
        screen_size=96,
        grayscale_obs=False,
        grayscale_newaxis=False,
        scale_obs=True,
    )
    env = FrameStackObservation(env, stack_size=4)

    return env

In [ ]:
# -- Define Model


class ResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int | str = "same",
        device: str = "cpu",
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            device=device,
        )
        self.ln1 = nn.InstanceNorm2d(out_channels, device=device)
        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
            device=device,
        )
        self.ln2 = nn.InstanceNorm2d(out_channels, device=device)

    def forward(self, x: torch.Tensor):
        y = self.conv1(x)
        y = self.ln1(x)
        y = F.relu(y)
        y = self.conv2(y)
        y = self.ln2(x)
        y = y + x
        y = F.relu(y)
        return y


class AtariObservationNetwork(nn.Module):
    def __init__(
        self,
        in_channels: int,
        kernel_size: int = 3,
        device: str = "cpu",
    ):
        super().__init__()

        self.down_conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=128,
            kernel_size=kernel_size,
            stride=2,
            padding=1,
            device=device,
        )

        self.res_blocks1 = nn.Sequential(
            *[
                ResidualBlock(128, 128, kernel_size, padding="same", device=device)
                for _ in range(2)
            ]
        )

        self.down_conv2 = nn.Conv2d(
            in_channels=128,
            out_channels=256,
            kernel_size=kernel_size,
            stride=2,
            padding=1,
            device=device,
        )

        self.res_blocks2 = nn.Sequential(
            *[
                ResidualBlock(256, 256, kernel_size, padding="same", device=device)
                for _ in range(3)
            ]
        )

        self.avg_pool1 = nn.AvgPool2d(kernel_size=kernel_size, stride=2, padding=1)

        self.res_blocks3 = nn.Sequential(
            *[
                ResidualBlock(256, 256, kernel_size, padding="same", device=device)
                for _ in range(3)
            ]
        )

        self.avg_pool2 = nn.AvgPool2d(kernel_size=kernel_size, stride=2, padding=1)

    def forward(self, x):
        x = self.down_conv1(x)
        x = self.res_blocks1(x)
        x = self.down_conv2(x)
        x = self.res_blocks2(x)
        x = self.avg_pool1(x)
        x = self.res_blocks3(x)
        x = self.avg_pool2(x)
        return x


class RepresentationNetwork(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        num_blocks: int = 4,
        kernel_size: int = 3,
        device: str = "cpu",
    ):
        super().__init__()

        self.atari_downsample = AtariObservationNetwork(
            in_channels=in_channels, kernel_size=kernel_size, device=device
        )

        self.res_blocks = nn.Sequential(
            *[
                ResidualBlock(
                    hidden_channels, hidden_channels, kernel_size, device=device
                )
                for _ in range(num_blocks)
            ]
        )

    def forward(self, x):
        x = self.atari_downsample(x)
        x = self.res_blocks(x)
        return x


class DynamicsNetwork(nn.Module):
    def __init__(
        self,
        action_space: int,
        hidden_channels: int,
        num_blocks: int = 16,
        kernel_size: int = 3,
        device: str = "cpu",
    ):
        super().__init__()
        self.action_space = action_space
        in_channels = hidden_channels + 1

        # -- New Representation
        blocks = [
            nn.Conv2d(
                in_channels, hidden_channels, kernel_size, padding="same", device=device
            ),
        ]

        for _ in range(num_blocks - 1):
            blocks.append(
                ResidualBlock(
                    hidden_channels, hidden_channels, kernel_size, device=device
                )
            )

        self.dyn_res_blocks = nn.Sequential(*blocks)

    def forward(self, hidden_state, action):
        # action -> [...]
        device = hidden_state.device
        action = torch.tensor(action).to(device)
        B, C, H, W = hidden_state.shape
        action_one_hot = torch.ones((B, 1, H, W)).float().to(device)
        action_one_hot = (
            action[:, None, None, None] * action_one_hot / self.action_space
        )

        x = torch.cat((hidden_state, action_one_hot), dim=1)

        new_hidden_state = self.dyn_res_blocks(x)

        return new_hidden_state


class PolicyNetwork(nn.Module):
    def __init__(self, hidden_channels: int, kernel_size: int = 3, device: str = "cpu"):
        super().__init__()

        self.pred_head = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_channels,
                out_channels=32,
                kernel_size=kernel_size,
                padding="same",
                device=device,
            ),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(in_features=(32 * 6 * 6), out_features=17, device=device),
        )

    def forward(self, x):
        return self.pred_head(x)


class ValueNetwork(nn.Module):
    def __init__(
        self,
        hidden_channels: int,
        support_size: int,
        kernel_size: int = 3,
        device: str = "cpu",
    ):
        super().__init__()

        self.value_head = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_channels,
                out_channels=32,
                kernel_size=kernel_size,
                padding="same",
                device=device,
            ),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(
                in_features=(32 * 6 * 6), out_features=support_size, device=device
            ),
        )

    def forward(self, x):
        return self.value_head(x)

In [ ]:
# -- Define Gumbel MuZero Interface


class AtariMuZero(AbstractNetwork):
    def __init__(
        self,
        num_frames: int = 4,
        hidden_channels: int = 256,
        kernel_size: int = 3,
        rep_blocks: int = 4,
        dyn_blocks: int = 16,
        device: str = "cpu",
    ):
        super().__init__()

        self.representation_network = RepresentationNetwork(
            in_channels=num_frames * 3,
            hidden_channels=hidden_channels,
            num_blocks=rep_blocks,
            kernel_size=kernel_size,
            device=device,
        )
        self.dynamics_network = DynamicsNetwork(
            hidden_channels=hidden_channels,
            action_space=17,
            num_blocks=dyn_blocks,
            kernel_size=kernel_size,
            device=device,
        )
        self.policy_network = PolicyNetwork(
            hidden_channels=hidden_channels, device=device
        )
        self.value_network = ValueNetwork(
            hidden_channels=hidden_channels, support_size=601, device=device
        )
        self.reward_network = ValueNetwork(
            hidden_channels=hidden_channels, support_size=601, device=device
        )

        map_min = torch.tensor(-300.0, dtype=torch.float32)
        map_max = torch.tensor(300.0, dtype=torch.float32)
        self.value_encoder = ValueEncoder(
            *tuple(map(inverse_contractive_mapping, (map_min, map_max))),
            0,
        )
        self.reward_encoder = ValueEncoder(
            *tuple(map(inverse_contractive_mapping, (map_min, map_max))),
            0,
        )

    def initial_inference(self, x, training=False):
        batch_size = x.shape[0]

        hidden_state = self.representation_network(x)
        value_logits = self.value_network(hidden_state)
        policy_logits = self.policy_network(hidden_state)
        reward = torch.zeros((batch_size,))
        value = self.value_encoder.decode_softmax(value_logits)

        outputs = {
            "value": value,
            "value_logits": value_logits,
            "reward": reward,
            "policy_logits": policy_logits,
            "hidden_state": hidden_state,
        }

        return outputs

    def recurrent_inference(self, encoded_state, action, training=False):
        next_hidden_state = self.dynamics_network(encoded_state, action)
        reward_logits = self.reward_network(next_hidden_state)

        value_logits = self.value_network(next_hidden_state)
        policy_logits = self.policy_network(next_hidden_state)

        reward = self.reward_encoder.decode_softmax(reward_logits)
        value = self.value_encoder.decode_softmax(value_logits)

        outputs = {
            "value": value,
            "value_logits": value_logits,
            "reward": reward,
            "reward_logits": reward_logits,
            "policy_logits": policy_logits,
            "hidden_state": next_hidden_state,
        }

        return outputs


In [ ]:
# -- Observation Utilities
def create_breakout_action_mask() -> torch.Tensor:
    action_mask = torch.zeros((1, 17))
    # only the first 4 actions are legal
    action_mask[:, :4] = 1
    return action_mask


def preprocess_obs(obs: np.ndarray, device: str = "cpu"):
    N, H, W, C = obs.shape
    torch_obs = (
        torch.from_numpy(obs)
        .reshape(H, W, N * C)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )
    action_mask = create_breakout_action_mask().to(device)

    return {"observations": torch_obs, "action_mask": action_mask}


In [ ]:
# -- Create Test Model
device = config.DEVICE
network = AtariMuZero(device=device)
gumbel = GumbelMuZero(network, model_config)

In [ ]:
# -- Create test env
env = breakout_env()
obs, info = env.reset()
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
env.close()

In [ ]:
# -- Test Network
broadcasted_obs = preprocess_obs(obs)["observations"].broadcast_to((8, 12, 96, 96))
# initial_output = network.initial_inference(preprocess_obs(obs)["observations"])
initial_output = network.initial_inference(broadcasted_obs)
actions = np.array([1, 2, 3, 4, 5, 6, 7, 8])
recurrent_output = network.recurrent_inference(initial_output["hidden_state"], actions)

In [ ]:
# -- Test Gumbel
actions, target_policies, roots_values = gumbel.policy(data=preprocess_obs(obs))
print(actions)
print(target_policies)
print(roots_values)

In [ ]:
from TestInterface.test_data_worker import DataWorker
from TestInterface.test_global_buffer import GlobalBuffer
from TestInterface.test_replay_muzero_buffer import ReplayBuffer

from Models.GumbelModels.gumbel_trainer import Trainer
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime


def rollout(agent: GumbelMuZero, buffer: ReplayBuffer):
    buffer.reset()
    obs, info = env.reset()
    terminated, truncated = False, False

    while not (terminated or truncated):
        # Get action
        current_obs = preprocess_obs(obs, device)
        actions, target_policies, roots_values = agent.policy(data=current_obs)

        # There's only one env running
        action = actions[0]
        policy = target_policies[0]
        root_value = roots_values[0]

        # Step in Environment
        next_obs, reward, terminated, truncated, info = env.step(action)

        # Add to buffer
        current_obs["observations"].squeeze_()
        buffer.store_gumbel_buffer(current_obs, action, reward, policy, root_value)

        obs = next_obs

    buffer.store_global_position_buffer()

In [ ]:
# -- Create SummaryWriter
current_time = datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = "logs/gradient_tape/" + current_time + "/train"
summary_writer = SummaryWriter(train_log_dir)

# Create buffer
global_buffer = GlobalBuffer()
buffer = ReplayBuffer(global_buffer, key="breakout")

# -- Create Model
device = config.DEVICE
network = AtariMuZero(device=device)
gumbel = GumbelMuZero(network, model_config)

# -- Load from timestep
time_step = config.STARTING_EPISODE
network.tft_load_model(episode=time_step)

# -- Create Trainer
trainer = Trainer(network, summary_writer)


In [ ]:
while True:
    rollout(gumbel, buffer)

    while global_buffer.available_batch():
        batch = global_buffer.sample_position_batch()
        trainer.train_network(batch, time_step)
        time_step += 1

        if time_step % 100 == 0:
            network.tft_save_model(time_step, trainer.optimizer)